# Notebook Bukti Final UAS Data Analytics
## Segmentasi Pola Transaksi Pelanggan Toko Online Menggunakan K-Means Clustering

Notebook ini adalah versi terpadu untuk membuktikan proses **BAB II sampai BAB VI**:

- BAB II: Data Understanding
- BAB III: Data Preparation
- BAB IV: Modeling
- BAB V: Evaluation
- BAB VI: Deployment

**Cara pakai di Google Colab:** upload file `customer_spending_1M_2018_2025.csv`, lalu jalankan semua cell dari atas sampai bawah. Notebook akan menghasilkan seluruh file CSV, PNG, PKL, dan file Streamlit yang diperlukan untuk laporan dan demo.

In [ ]:
# =========================================================
# SETUP GLOBAL
# =========================================================
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

RANDOM_STATE = 42
print("Library berhasil dimuat.")

# BAB II — Data Understanding
Tahap ini membaca dataset, mengecek struktur data, eksplorasi statistik, distribusi kategori, missing value, kualitas tanggal, duplikasi, dan outlier awal.

In [ ]:
# =========================================================
# KODE 2.1 MEMUAT DATASET DAN MEMERIKSA DIMENSI DATA
# =========================================================
dataset_path = "customer_spending_1M_2018_2025.csv"

if not os.path.exists(dataset_path):
    local_path = "/mnt/data/customer_spending_1M_2018_2025.csv"
    if os.path.exists(local_path):
        dataset_path = local_path

if not os.path.exists(dataset_path):
    try:
        from google.colab import files
        print("Upload file customer_spending_1M_2018_2025.csv")
        uploaded = files.upload()
        dataset_path = list(uploaded.keys())[0]
    except Exception:
        raise FileNotFoundError("Dataset belum ditemukan. Upload customer_spending_1M_2018_2025.csv terlebih dahulu.")

df = pd.read_csv(dataset_path)
shape_awal = df.shape
print("Dataset berhasil dimuat.")
print("Ukuran dataset:", df.shape)
display(df.head())

In [ ]:
# =========================================================
# KODE 2.2 PEMERIKSAAN TIPE DATA ATRIBUT
# =========================================================
print("Tipe data atribut:")
display(df.dtypes.to_frame("Tipe Data"))

info_dataset = pd.DataFrame({
    "Komponen": ["Nama Dataset", "Sumber", "Jumlah Baris", "Jumlah Kolom", "Periode Data", "Jenis Data", "Teknik Data Mining", "Algoritma"],
    "Keterangan": ["Online Store Customer Transactions (1M Rows)", "Kaggle", f"{df.shape[0]:,}", df.shape[1], "2018-2025", "Data transaksi pelanggan toko online", "Clustering", "K-Means Clustering"]
})
display(info_dataset)

In [ ]:
# =========================================================
# KODE 2.3 STATISTIK DESKRIPTIF ATRIBUT NUMERIK
# =========================================================
kolom_numerik_awal = ["Age", "Referral", "Amount_spent"]
statistik_numerik = df[kolom_numerik_awal].describe().round(2)
display(statistik_numerik)

In [ ]:
# =========================================================
# KODE 2.4 DISTRIBUSI ATRIBUT KATEGORIKAL
# =========================================================
kolom_kategorikal_awal = ["Gender", "Marital_status", "Employees_status", "Payment_method", "Segment"]
for kolom in kolom_kategorikal_awal:
    print(f"\nDistribusi kategori: {kolom}")
    display(df[kolom].value_counts(dropna=False).to_frame("Jumlah"))

In [ ]:
# =========================================================
# KODE 2.5 PEMERIKSAAN KUALITAS DATA AWAL
# =========================================================
missing_awal = pd.DataFrame({
    "Missing Value": df.isnull().sum(),
    "Persentase (%)": (df.isnull().sum() / len(df) * 100).round(2)
})
missing_awal = missing_awal[missing_awal["Missing Value"] > 0].sort_values("Missing Value", ascending=False)
print("Missing value awal:")
display(missing_awal)

plt.figure(figsize=(8, 4))
missing_awal["Missing Value"].plot(kind="bar")
plt.title("Jumlah Missing Value per Atribut")
plt.xlabel("Atribut")
plt.ylabel("Jumlah Missing Value")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("fig_missing_values.png", dpi=180)
plt.show()

validasi_awal = {
    "Baris duplikat identik": int(df.duplicated().sum()),
    "Transaction_ID unik": int(df["Transaction_ID"].nunique()),
    "Transaction_ID duplikat": int(df["Transaction_ID"].duplicated().sum()),
}

df["Transaction_date_parsed"] = pd.to_datetime(df["Transaction_date"], errors="coerce")
validasi_awal["Tanggal gagal parsing"] = int(df["Transaction_date_parsed"].isna().sum())
validasi_awal["Tanggal awal"] = df["Transaction_date_parsed"].min()
validasi_awal["Tanggal akhir"] = df["Transaction_date_parsed"].max()
validasi_awal["Jumlah State_names"] = int(df["State_names"].nunique(dropna=True))
validasi_awal["Segment bernilai Missing"] = int((df["Segment"] == "Missing").sum())
validasi_awal["Nilai unik Referral"] = sorted(df["Referral"].dropna().unique().tolist())

display(pd.DataFrame(list(validasi_awal.items()), columns=["Pemeriksaan", "Hasil"]))

In [ ]:
# =========================================================
# KODE 2.6 PEMERIKSAAN OUTLIER AWAL DENGAN IQR DAN BOXPLOT
# =========================================================
def cek_outlier_iqr(dataframe, kolom):
    data = dataframe[kolom].dropna()
    q1 = data.quantile(0.25)
    q3 = data.quantile(0.75)
    iqr = q3 - q1
    batas_bawah = q1 - 1.5 * iqr
    batas_atas = q3 + 1.5 * iqr
    jumlah_outlier = ((data < batas_bawah) | (data > batas_atas)).sum()
    return {
        "Atribut": kolom,
        "Q1": round(q1, 2),
        "Q3": round(q3, 2),
        "IQR": round(iqr, 2),
        "Batas Bawah": round(batas_bawah, 2),
        "Batas Atas": round(batas_atas, 2),
        "Jumlah Outlier": int(jumlah_outlier)
    }

outlier_awal = pd.DataFrame([cek_outlier_iqr(df, "Age"), cek_outlier_iqr(df, "Amount_spent")])
display(outlier_awal)

plt.figure(figsize=(10, 4))
plt.boxplot(df["Amount_spent"].dropna(), vert=False)
plt.title("Boxplot Distribusi Pengeluaran (Amount_spent)")
plt.xlabel("Nilai Amount_spent")
plt.grid(axis="x", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.savefig("fig_amount_boxplot.png", dpi=180)
plt.show()

# BAB III — Data Preparation
Tahap ini menyiapkan data agar dapat digunakan oleh K-Means: seleksi atribut, missing value, validasi anomali, konsistensi kategori, transformasi tanggal, one-hot encoding, standardisasi, dan penyimpanan dataset hasil preparation.

In [ ]:
# =========================================================
# KODE 3.1 SELEKSI ATRIBUT UNTUK CLUSTERING
# =========================================================
fitur_model = ["Age", "Gender", "Marital_status", "Employees_status", "Payment_method", "Referral", "Amount_spent"]
fitur_profiling = ["Segment", "State_names", "Transaction_date"]
fitur_dibuang = ["Transaction_ID"]

fitur_tabel = pd.DataFrame({
    "Kelompok": ["Modeling"] * len(fitur_model) + ["Profiling"] * len(fitur_profiling) + ["Dibuang"] * len(fitur_dibuang),
    "Atribut": fitur_model + fitur_profiling + fitur_dibuang
})
display(fitur_tabel)

In [ ]:
# =========================================================
# KODE 3.2 PENANGANAN MISSING VALUE DAN PEMBERSIHAN DATA
# =========================================================
df_clean = df.copy()

# Amount_spent adalah fitur inti, sehingga baris kosong pada kolom ini dihapus.
jumlah_awal = len(df_clean)
df_clean = df_clean.dropna(subset=["Amount_spent"]).copy()
jumlah_setelah_drop = len(df_clean)

median_age = df_clean["Age"].median()
modus_referral = df_clean["Referral"].mode(dropna=True).iloc[0]

df_clean["Age"] = df_clean["Age"].fillna(median_age)
df_clean["Referral"] = df_clean["Referral"].fillna(modus_referral)
df_clean["Gender"] = df_clean["Gender"].fillna("Unknown")
df_clean["Employees_status"] = df_clean["Employees_status"].fillna("Unknown")

print("Jumlah data awal:", jumlah_awal)
print("Jumlah data setelah Amount_spent kosong dihapus:", jumlah_setelah_drop)
print("Jumlah baris terhapus:", jumlah_awal - jumlah_setelah_drop)
print("Median Age untuk imputasi:", median_age)
print("Modus Referral untuk imputasi:", modus_referral)
print("Missing value akhir pada fitur model:")
display(df_clean[fitur_model].isnull().sum().to_frame("Missing"))

In [ ]:
# =========================================================
# KODE 3.3 PEMERIKSAAN PSEUDO-MISSING, OUTLIER, DAN KONSISTENSI KATEGORI
# =========================================================
print("Distribusi Segment setelah pembersihan utama:")
display(df_clean["Segment"].value_counts(dropna=False).to_frame("Jumlah"))

outlier_setelah = pd.DataFrame([cek_outlier_iqr(df_clean, "Age"), cek_outlier_iqr(df_clean, "Amount_spent")])
print("Outlier setelah pembersihan utama:")
display(outlier_setelah)

kolom_string = ["Gender", "Marital_status", "State_names", "Segment", "Employees_status", "Payment_method"]
for kolom in kolom_string:
    df_clean[kolom] = df_clean[kolom].astype(str).str.strip()

mapping_employees = {
    "workers": "Workers",
    "self-employed": "Self-Employed",
    "Employees": "Employees",
    "Unemployment": "Unemployment",
    "Unknown": "Unknown"
}
df_clean["Employees_status"] = df_clean["Employees_status"].replace(mapping_employees)

print("Distribusi Employees_status setelah konsistensi kategori:")
display(df_clean["Employees_status"].value_counts(dropna=False).to_frame("Jumlah"))

In [ ]:
# =========================================================
# KODE 3.4 TRANSFORMASI TANGGAL, ENCODING, STANDARDISASI, DAN SIMPAN PREPROCESSOR
# =========================================================
df_clean["Transaction_date"] = pd.to_datetime(df_clean["Transaction_date"], errors="coerce")
df_clean["Year"] = df_clean["Transaction_date"].dt.year
df_clean["Month"] = df_clean["Transaction_date"].dt.month
df_clean["DayOfWeek"] = df_clean["Transaction_date"].dt.day_name()

fitur_numerik = ["Age", "Referral", "Amount_spent"]
fitur_kategorikal = ["Gender", "Marital_status", "Employees_status", "Payment_method"]
X_raw = df_clean[fitur_numerik + fitur_kategorikal].copy()

try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), fitur_numerik),
        ("cat", encoder, fitur_kategorikal)
    ],
    remainder="drop"
)

X_clustering_array = preprocessor.fit_transform(X_raw)
nama_fitur_akhir = preprocessor.get_feature_names_out()
X_clustering = pd.DataFrame(X_clustering_array, columns=nama_fitur_akhir)

joblib.dump(preprocessor, "preprocessor_clustering.pkl")

print("Ukuran dataset akhir untuk clustering:", X_clustering.shape)
print("Total missing pada dataset akhir:", int(X_clustering.isnull().sum().sum()))
print("Preprocessor disimpan sebagai preprocessor_clustering.pkl")
display(X_clustering.head())

In [ ]:
# =========================================================
# KODE 3.5 EKSPOR DATASET HASIL DATA PREPARATION
# =========================================================
X_clustering.to_csv("data_clustering_prepared.csv", index=False)

data_profiling = df_clean[[
    "Transaction_ID", "Transaction_date", "Age", "Gender", "Marital_status",
    "Employees_status", "Payment_method", "Referral", "Amount_spent",
    "Segment", "State_names", "Year", "Month", "DayOfWeek"
]].copy()

data_profiling.to_csv("data_clustering_profile.csv", index=False)

ringkasan_akhir_bab3 = pd.DataFrame({
    "Komponen": ["Data awal", "Data akhir", "Fitur sebelum encoding", "Fitur setelah encoding", "Target/label", "Train-test split"],
    "Hasil": [f"{shape_awal[0]:,} baris x {shape_awal[1]} kolom", f"{X_clustering.shape[0]:,} baris x {X_clustering.shape[1]} fitur", len(fitur_model), X_clustering.shape[1], "Tidak ada", "Tidak dilakukan"]
})
display(ringkasan_akhir_bab3)
print("File berhasil dibuat: data_clustering_prepared.csv, data_clustering_profile.csv")

# BAB IV — Modeling
Tahap ini menentukan jumlah cluster optimal, melatih model K-Means final, membuat visualisasi awal, dan menyimpan output modeling.

In [ ]:
# =========================================================
# KODE 4.1 PENENTUAN KANDIDAT JUMLAH CLUSTER
# =========================================================
X = X_clustering.copy()
profile = data_profiling.copy()

X_sample = X.sample(n=min(100000, len(X)), random_state=RANDOM_STATE)
X_metric = X_sample.sample(n=min(10000, len(X_sample)), random_state=RANDOM_STATE)

hasil_k = []
for k in range(2, 11):
    model_k = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10, max_iter=300)
    model_k.fit(X_sample)
    label_metric = model_k.predict(X_metric)
    hasil_k.append({
        "k": k,
        "inertia": model_k.inertia_,
        "silhouette": silhouette_score(X_metric, label_metric, sample_size=min(5000, len(X_metric)), random_state=RANDOM_STATE),
        "davies_bouldin": davies_bouldin_score(X_metric, label_metric),
        "calinski_harabasz": calinski_harabasz_score(X_metric, label_metric)
    })

hasil_k = pd.DataFrame(hasil_k)
display(hasil_k)
hasil_k.to_csv("bab4_k_selection_results.csv", index=False)

In [ ]:
# =========================================================
# KODE 4.2 VISUALISASI ELBOW, SILHOUETTE, DAN DAVIES-BOULDIN
# =========================================================
plt.figure(figsize=(8, 5))
plt.plot(hasil_k["k"], hasil_k["inertia"], marker="o")
plt.title("Elbow Method untuk Penentuan Jumlah Cluster")
plt.xlabel("Jumlah Cluster (K)")
plt.ylabel("Inertia / WCSS")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig("fig_bab4_elbow.png", dpi=180)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(hasil_k["k"], hasil_k["silhouette"], marker="o")
plt.title("Silhouette Analysis untuk Penentuan Jumlah Cluster")
plt.xlabel("Jumlah Cluster (K)")
plt.ylabel("Silhouette Score")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig("fig_bab4_silhouette.png", dpi=180)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(hasil_k["k"], hasil_k["davies_bouldin"], marker="o")
plt.title("Davies-Bouldin Index untuk Alternatif Jumlah Cluster")
plt.xlabel("Jumlah Cluster (K)")
plt.ylabel("Davies-Bouldin Index")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig("fig_bab4_dbi.png", dpi=180)
plt.show()

In [ ]:
# =========================================================
# KODE 4.3 PELATIHAN MODEL K-MEANS AKHIR K=4
# =========================================================
k_final = 4
kmeans_final = KMeans(n_clusters=k_final, random_state=RANDOM_STATE, n_init=10, max_iter=300)
cluster_labels = kmeans_final.fit_predict(X)

print("Jumlah cluster akhir:", k_final)
print("Inertia/WCSS model akhir:", kmeans_final.inertia_)
print("Distribusi cluster:")
print(pd.Series(cluster_labels).value_counts().sort_index())

In [ ]:
# =========================================================
# KODE 4.4 EVALUASI INTERNAL MODEL AKHIR DAN DISTRIBUSI CLUSTER
# =========================================================
eval_idx = X.sample(n=min(10000, len(X)), random_state=RANDOM_STATE).index
X_eval = X.loc[eval_idx]
labels_eval = cluster_labels[eval_idx]

final_metrics = pd.DataFrame([{
    "Jumlah Cluster (K)": k_final,
    "Inertia/WCSS": kmeans_final.inertia_,
    "Silhouette Score (sample 10.000)": silhouette_score(X_eval, labels_eval, sample_size=min(5000, len(X_eval)), random_state=RANDOM_STATE),
    "Davies-Bouldin Index (sample 10.000)": davies_bouldin_score(X_eval, labels_eval),
    "Calinski-Harabasz Index (sample 10.000)": calinski_harabasz_score(X_eval, labels_eval)
}])
display(final_metrics)
final_metrics.to_csv("bab4_final_model_metrics.csv", index=False)

profile_cluster = profile.copy()
profile_cluster["Cluster"] = cluster_labels

cluster_size = profile_cluster["Cluster"].value_counts().sort_index().rename("Jumlah").to_frame()
cluster_size["Persentase (%)"] = (cluster_size["Jumlah"] / len(profile_cluster) * 100).round(2)
display(cluster_size)
cluster_size.to_csv("bab4_cluster_size.csv")

plt.figure(figsize=(8, 5))
plt.bar(cluster_size.index.astype(str), cluster_size["Jumlah"])
plt.title("Jumlah Transaksi pada Setiap Cluster")
plt.xlabel("Cluster")
plt.ylabel("Jumlah Transaksi")
plt.tight_layout()
plt.savefig("fig_bab4_cluster_size.png", dpi=180)
plt.show()

In [ ]:
# =========================================================
# KODE 4.5 RINGKASAN AWAL PROFIL CLUSTER, PCA, DAN SIMPAN OUTPUT MODELING
# =========================================================
ringkasan = []
for c in sorted(profile_cluster["Cluster"].unique()):
    data_c = profile_cluster[profile_cluster["Cluster"] == c]
    ringkasan.append({
        "Cluster": c,
        "Jumlah Transaksi": len(data_c),
        "Persentase (%)": round(len(data_c) / len(profile_cluster) * 100, 2),
        "Rata-rata Age": round(data_c["Age"].mean(), 2),
        "Median Age": round(data_c["Age"].median(), 2),
        "Rata-rata Amount_spent": round(data_c["Amount_spent"].mean(), 2),
        "Median Amount_spent": round(data_c["Amount_spent"].median(), 2),
        "Referral dominan": data_c["Referral"].mode().iat[0],
        "Gender dominan": data_c["Gender"].mode().iat[0],
        "Marital dominan": data_c["Marital_status"].mode().iat[0],
        "Employees dominan": data_c["Employees_status"].mode().iat[0],
        "Payment dominan": data_c["Payment_method"].mode().iat[0],
        "Segment bawaan dominan": data_c["Segment"].mode().iat[0],
        "State dominan": data_c["State_names"].mode().iat[0]
    })

cluster_profile_summary = pd.DataFrame(ringkasan)
display(cluster_profile_summary)
cluster_profile_summary.to_csv("bab4_cluster_profile_summary.csv", index=False)

viz_idx = X.sample(n=min(10000, len(X)), random_state=RANDOM_STATE).index
pca = PCA(n_components=2, random_state=RANDOM_STATE)
pcs = pca.fit_transform(X.loc[viz_idx])

viz = pd.DataFrame({"PC1": pcs[:, 0], "PC2": pcs[:, 1], "Cluster": cluster_labels[viz_idx].astype(str)})
plt.figure(figsize=(8, 6))
for cl in sorted(viz["Cluster"].unique()):
    sub = viz[viz["Cluster"] == cl]
    plt.scatter(sub["PC1"], sub["PC2"], s=8, alpha=0.5, label=f"Cluster {cl}")
plt.title("Visualisasi Cluster Menggunakan PCA 2D (Sample 10.000)")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.2f}% var.)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.2f}% var.)")
plt.legend(markerscale=2)
plt.tight_layout()
plt.savefig("fig_bab4_pca_cluster.png", dpi=180)
plt.show()

profile_cluster.to_csv("data_clustering_profile_with_cluster.csv", index=False)
pd.DataFrame(kmeans_final.cluster_centers_, columns=X.columns).to_csv("bab4_kmeans_centroids.csv", index=False)
joblib.dump(kmeans_final, "model_kmeans_clustering_k4.pkl")

print("Output modeling berhasil disimpan.")

# BAB V — Evaluation
Tahap ini mengevaluasi metrik internal, visualisasi cluster, dan profiling karakteristik setiap cluster.

In [ ]:
# =========================================================
# KODE 5.1 RINGKASAN METRIK INTERNAL
# =========================================================
internal_metrics = pd.DataFrame({
    "Metrik": ["Inertia/WCSS", "Silhouette Score (sample 10.000)", "Davies-Bouldin Index (sample 10.000)", "Calinski-Harabasz Index (sample 10.000)"],
    "Nilai": [
        float(final_metrics["Inertia/WCSS"].iloc[0]),
        float(final_metrics["Silhouette Score (sample 10.000)"].iloc[0]),
        float(final_metrics["Davies-Bouldin Index (sample 10.000)"].iloc[0]),
        float(final_metrics["Calinski-Harabasz Index (sample 10.000)"].iloc[0])
    ],
    "Interpretasi Singkat": [
        "Semakin kecil semakin baik untuk kompaksi cluster pada K-Means.",
        "Nilai mendekati 1 semakin baik; hasil menunjukkan pemisahan cluster belum sangat kuat.",
        "Semakin kecil semakin baik; digunakan sebagai salah satu dasar memilih K=4.",
        "Semakin besar semakin baik; digunakan sebagai metrik pendukung."
    ]
})
display(internal_metrics)
internal_metrics.to_csv("bab5_internal_metrics_summary.csv", index=False)

In [ ]:
# =========================================================
# KODE 5.2 VISUALISASI CLUSTER UNTUK EVALUATION
# =========================================================
plt.figure(figsize=(8, 5))
plt.bar(cluster_size.index.astype(str), cluster_size["Jumlah"])
plt.title("Jumlah Transaksi pada Setiap Cluster")
plt.xlabel("Cluster")
plt.ylabel("Jumlah Transaksi")
plt.tight_layout()
plt.savefig("fig_bab5_cluster_size.png", dpi=180)
plt.show()

amount_summary = profile_cluster.groupby("Cluster")["Amount_spent"].mean().round(2)
plt.figure(figsize=(8, 5))
plt.bar(amount_summary.index.astype(str), amount_summary.values)
plt.title("Rata-rata Amount_spent per Cluster")
plt.xlabel("Cluster")
plt.ylabel("Rata-rata Amount_spent")
plt.tight_layout()
plt.savefig("fig_bab5_avg_amount.png", dpi=180)
plt.show()

age_summary = profile_cluster.groupby("Cluster")["Age"].mean().round(2)
plt.figure(figsize=(8, 5))
plt.bar(age_summary.index.astype(str), age_summary.values)
plt.title("Rata-rata Age per Cluster")
plt.xlabel("Cluster")
plt.ylabel("Rata-rata Age")
plt.tight_layout()
plt.savefig("fig_bab5_avg_age.png", dpi=180)
plt.show()

# PCA untuk BAB V memakai sampel yang sama konsepnya dengan BAB IV
plt.figure(figsize=(8, 6))
for cl in sorted(viz["Cluster"].unique()):
    sub = viz[viz["Cluster"] == cl]
    plt.scatter(sub["PC1"], sub["PC2"], s=8, alpha=0.5, label=f"Cluster {cl}")
plt.title("Visualisasi Cluster Menggunakan PCA 2D (Sample 10.000)")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.2f}% var.)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.2f}% var.)")
plt.legend(markerscale=2)
plt.tight_layout()
plt.savefig("fig_bab5_pca_cluster.png", dpi=180)
plt.show()

In [ ]:
# =========================================================
# KODE 5.3 PROFILING NUMERIK DAN KATEGORIKAL SETIAP CLUSTER
# =========================================================
numeric_profile = profile_cluster.groupby("Cluster").agg(
    Jumlah_Transaksi=("Transaction_ID", "count"),
    Rata_rata_Age=("Age", "mean"),
    Median_Age=("Age", "median"),
    Rata_rata_Amount_spent=("Amount_spent", "mean"),
    Median_Amount_spent=("Amount_spent", "median"),
    Rata_rata_Referral=("Referral", "mean")
).round(2).reset_index()

display(numeric_profile)
numeric_profile.to_csv("bab5_numeric_profile_summary.csv", index=False)

def mode_value(s):
    return s.mode().iat[0] if not s.mode().empty else None

dominant_category = profile_cluster.groupby("Cluster").agg(
    Gender_Dominan=("Gender", mode_value),
    Marital_Dominan=("Marital_status", mode_value),
    Employees_Dominan=("Employees_status", mode_value),
    Payment_Dominan=("Payment_method", mode_value),
    Segment_Bawaan_Dominan=("Segment", mode_value),
    State_Dominan=("State_names", mode_value)
).reset_index()

display(dominant_category)
dominant_category.to_csv("bab5_dominant_category_profile.csv", index=False)

In [ ]:
# =========================================================
# KODE 5.4 INTERPRETASI DAN REKOMENDASI AWAL TIAP CLUSTER
# =========================================================
interpretation = pd.DataFrame([
    [0, "Transaksi high value dengan referral", "Pengeluaran tertinggi dan seluruhnya didominasi referral", "Pertahankan loyalitas, tawarkan program premium atau bundling produk"],
    [1, "Transaksi pelanggan muda berpengeluaran rendah", "Usia rata-rata paling muda dengan pengeluaran rendah", "Gunakan voucher, cashback, dan rekomendasi produk terjangkau"],
    [2, "Transaksi non-referral berpengeluaran sedang", "Cluster terbesar dan didominasi transaksi tanpa referral", "Dorong program referral dan retensi karena kelompok ini besar"],
    [3, "Transaksi usia senior berpengeluaran rendah", "Usia rata-rata paling tinggi dengan pengeluaran rendah", "Gunakan pendekatan layanan sederhana dan promo kebutuhan rutin"]
], columns=["Cluster", "Nama Profil", "Karakteristik Utama", "Rekomendasi Awal"])

display(interpretation)
interpretation.to_csv("bab5_cluster_interpretation_summary.csv", index=False)
print("Output evaluation berhasil disimpan.")

# BAB VI — Deployment
Tahap ini membuat artefak deployment: model, preprocessor, aplikasi Streamlit, requirements, README, contoh prediksi input baru, dan manifest output.

In [ ]:
# =========================================================
# KODE 6.1 CONTOH PREDIKSI CLUSTER UNTUK INPUT BARU
# =========================================================
# Input baru masih mentah sehingga harus diproses dengan preprocessor BAB III.
input_baru = pd.DataFrame([{
    "Age": 35,
    "Gender": "Female",
    "Marital_status": "Married",
    "Employees_status": "Employees",
    "Payment_method": "PayPal",
    "Referral": 1.0,
    "Amount_spent": 1000.0
}])

input_prepared_array = preprocessor.transform(input_baru)
input_prepared = pd.DataFrame(input_prepared_array, columns=preprocessor.get_feature_names_out())
cluster_prediksi = int(kmeans_final.predict(input_prepared)[0])

print("Input baru:")
display(input_baru)
print("Input setelah preprocessing:")
display(input_prepared)
print("Prediksi cluster:", cluster_prediksi)
print("Interpretasi cluster:")
display(interpretation[interpretation["Cluster"] == cluster_prediksi])

In [ ]:
# =========================================================
# KODE 6.2 MEMBUAT FILE APLIKASI STREAMLIT
# =========================================================
app_code = r"""
import os
import pandas as pd
import streamlit as st
import joblib

st.set_page_config(page_title="Deployment K-Means Clustering", layout="wide")

@st.cache_resource
def load_artifacts():
    model = joblib.load("model_kmeans_clustering_k4.pkl")
    preprocessor = joblib.load("preprocessor_clustering.pkl")
    interpretation = pd.read_csv("bab5_cluster_interpretation_summary.csv")
    numeric_profile = pd.read_csv("bab5_numeric_profile_summary.csv")
    return model, preprocessor, interpretation, numeric_profile

st.title("Deployment Segmentasi Pola Transaksi Pelanggan")
st.write("Aplikasi ini menggunakan model K-Means K=4 untuk menentukan cluster transaksi baru.")

required_files = [
    "model_kmeans_clustering_k4.pkl",
    "preprocessor_clustering.pkl",
    "bab5_cluster_interpretation_summary.csv",
    "bab5_numeric_profile_summary.csv"
]
missing_files = [f for f in required_files if not os.path.exists(f)]
if missing_files:
    st.error("File deployment belum lengkap. Upload file berikut ke folder aplikasi:")
    st.write(missing_files)
    st.stop()

model, preprocessor, interpretation, numeric_profile = load_artifacts()

st.subheader("Input Data Transaksi Baru")
col1, col2, col3 = st.columns(3)

with col1:
    age = st.number_input("Age", min_value=15, max_value=78, value=35)
    gender = st.selectbox("Gender", ["Female", "Male", "Unknown"])
    marital_status = st.selectbox("Marital Status", ["Married", "Single"])

with col2:
    employees_status = st.selectbox("Employees Status", ["Employees", "Self-Employed", "Unemployment", "Workers", "Unknown"])
    payment_method = st.selectbox("Payment Method", ["PayPal", "Card", "Other"])
    referral = st.selectbox("Referral", [0.0, 1.0], format_func=lambda x: "Ya" if x == 1.0 else "Tidak")

with col3:
    amount_spent = st.number_input("Amount Spent", min_value=0.0, max_value=5000.0, value=1000.0, step=10.0)

input_data = pd.DataFrame([{
    "Age": age,
    "Gender": gender,
    "Marital_status": marital_status,
    "Employees_status": employees_status,
    "Payment_method": payment_method,
    "Referral": referral,
    "Amount_spent": amount_spent
}])

if st.button("Prediksi Cluster"):
    input_prepared_array = preprocessor.transform(input_data)
    input_prepared = pd.DataFrame(input_prepared_array, columns=preprocessor.get_feature_names_out())
    cluster = int(model.predict(input_prepared)[0])

    st.success(f"Transaksi masuk ke Cluster {cluster}")
    hasil = interpretation[interpretation["Cluster"] == cluster]
    profil = numeric_profile[numeric_profile["Cluster"] == cluster]

    if not hasil.empty:
        st.subheader("Interpretasi Cluster")
        st.write("**Nama Profil:**", hasil.iloc[0]["Nama Profil"])
        st.write("**Karakteristik Utama:**", hasil.iloc[0]["Karakteristik Utama"])
        st.write("**Rekomendasi Awal:**", hasil.iloc[0]["Rekomendasi Awal"])

    if not profil.empty:
        st.subheader("Profil Numerik Cluster")
        st.dataframe(profil, use_container_width=True)

st.subheader("Ringkasan Profil Semua Cluster")
st.dataframe(interpretation, use_container_width=True)
"""

with open("app_streamlit_clustering.py", "w", encoding="utf-8") as f:
    f.write(app_code)

with open("requirements_deployment.txt", "w", encoding="utf-8") as f:
    f.write("streamlit\npandas\nscikit-learn\njoblib\nmatplotlib\n")

readme_text = """Panduan Deployment Streamlit\n\nFile wajib dalam folder yang sama:\n1. app_streamlit_clustering.py\n2. model_kmeans_clustering_k4.pkl\n3. preprocessor_clustering.pkl\n4. bab5_cluster_interpretation_summary.csv\n5. bab5_numeric_profile_summary.csv\n6. requirements_deployment.txt\n\nCara menjalankan:\npip install -r requirements_deployment.txt\nstreamlit run app_streamlit_clustering.py\n\nCatatan:\npreprocessor_clustering.pkl wajib ada karena input baru harus diubah ke format 16 fitur numerik yang sama seperti data training.\n"""
with open("README_deployment.txt", "w", encoding="utf-8") as f:
    f.write(readme_text)

print("File deployment berhasil dibuat.")

In [ ]:
# =========================================================
# KODE 6.3 GAMBAR ALUR DEPLOYMENT DAN MANIFEST OUTPUT
# =========================================================
plt.figure(figsize=(10, 4))
plt.axis("off")
steps = [
    "Input transaksi baru",
    "Preprocessing\n(StandardScaler + One-Hot Encoding)",
    "Model K-Means K=4",
    "Cluster hasil prediksi",
    "Interpretasi & rekomendasi"
]
for i, step in enumerate(steps):
    x = 0.08 + i * 0.22
    plt.text(x, 0.55, step, ha="center", va="center", bbox=dict(boxstyle="round,pad=0.5", fc="white", ec="black"), fontsize=9)
    if i < len(steps)-1:
        plt.annotate("", xy=(x+0.1, 0.55), xytext=(x+0.15, 0.55), arrowprops=dict(arrowstyle="->", lw=1.5))
plt.title("Alur Deployment Hasil Clustering")
plt.tight_layout()
plt.savefig("fig_bab6_deployment_flow.png", dpi=180)
plt.show()

output_files = [
    ("fig_missing_values.png", "Visualisasi missing value BAB II"),
    ("fig_amount_boxplot.png", "Boxplot Amount_spent BAB II"),
    ("data_clustering_prepared.csv", "Dataset numerik siap modeling K-Means dari BAB III"),
    ("data_clustering_profile.csv", "Dataset bersih untuk profiling dari BAB III"),
    ("preprocessor_clustering.pkl", "Preprocessor untuk input baru pada deployment"),
    ("bab4_k_selection_results.csv", "Hasil uji K=2 sampai K=10"),
    ("bab4_final_model_metrics.csv", "Metrik model akhir K=4"),
    ("bab4_cluster_size.csv", "Jumlah transaksi per cluster"),
    ("bab4_cluster_profile_summary.csv", "Ringkasan awal profil cluster"),
    ("bab4_kmeans_centroids.csv", "Centroid model K-Means"),
    ("data_clustering_profile_with_cluster.csv", "Data profiling yang sudah memiliki kolom Cluster"),
    ("model_kmeans_clustering_k4.pkl", "Model K-Means final untuk deployment"),
    ("fig_bab4_elbow.png", "Grafik Elbow Method"),
    ("fig_bab4_silhouette.png", "Grafik Silhouette Analysis"),
    ("fig_bab4_dbi.png", "Grafik Davies-Bouldin Index"),
    ("fig_bab4_cluster_size.png", "Grafik ukuran cluster BAB IV"),
    ("fig_bab4_pca_cluster.png", "Visualisasi PCA BAB IV"),
    ("bab5_internal_metrics_summary.csv", "Ringkasan metrik internal BAB V"),
    ("bab5_numeric_profile_summary.csv", "Profil numerik cluster BAB V"),
    ("bab5_dominant_category_profile.csv", "Kategori dominan tiap cluster BAB V"),
    ("bab5_cluster_interpretation_summary.csv", "Interpretasi dan rekomendasi awal cluster"),
    ("fig_bab5_cluster_size.png", "Grafik jumlah transaksi per cluster BAB V"),
    ("fig_bab5_avg_amount.png", "Grafik rata-rata Amount_spent per cluster"),
    ("fig_bab5_avg_age.png", "Grafik rata-rata Age per cluster"),
    ("fig_bab5_pca_cluster.png", "Visualisasi PCA BAB V"),
    ("app_streamlit_clustering.py", "Aplikasi Streamlit untuk demo deployment"),
    ("requirements_deployment.txt", "Daftar library deployment"),
    ("README_deployment.txt", "Panduan menjalankan deployment"),
    ("fig_bab6_deployment_flow.png", "Gambar alur deployment")
]
manifest = pd.DataFrame(output_files, columns=["Nama File", "Kegunaan"])
manifest["Tersedia"] = manifest["Nama File"].apply(os.path.exists)
display(manifest)
manifest.to_csv("output_file_manifest.csv", index=False)

print("Manifest output disimpan sebagai output_file_manifest.csv")
print("Semua file utama tersedia:", manifest["Tersedia"].all())

## Selesai
Jika semua cell berhasil dijalankan, proyek sudah menghasilkan seluruh file bukti BAB II sampai BAB VI. File paling penting untuk demo adalah `app_streamlit_clustering.py`, `model_kmeans_clustering_k4.pkl`, `preprocessor_clustering.pkl`, `bab5_cluster_interpretation_summary.csv`, dan `bab5_numeric_profile_summary.csv`.